# H1B Data Analysis

--- Add Description

## Load Dataset

In [1]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

chmod: cannot access '/root/.kaggle/kaggle.json': Permission denied


In [ ]:
!kaggle datasets download -d zongaobian/h1b-lca-disclosure-data-2020-2024

Dataset URL: https://www.kaggle.com/datasets/zongaobian/h1b-lca-disclosure-data-2020-2024
License(s): CC0-1.0


In [2]:
! unzip h1b-lca-disclosure-data-2020-2024.zip

Archive:  h1b-lca-disclosure-data-2020-2024.zip
  inflating: Combined_LCA_Disclosure_Data_FY2020.csv  
  inflating: Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv  
  inflating: Combined_LCA_Disclosure_Data_FY2021.csv  
  inflating: Combined_LCA_Disclosure_Data_FY2022.csv  
  inflating: Combined_LCA_Disclosure_Data_FY2023.csv  
  inflating: Combined_LCA_Disclosure_Data_FY2024.csv  


In [2]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [3]:
sns.set_style("whitegrid")

In [4]:
data_path = "Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv"

In [ ]:
data = pd.read_csv(data_path, parse_dates=['RECEIVED_DATE'], low_memory=False)
data.head()

In [ ]:
data.info()

In [ ]:
data['EMPLOYER_STATE'].unique()

# Data Cleaning/Pre-Processing

## SOC Titles

There are way too many unique values in `SOC_TITLE` that makes it difficult to analyse.

We'll use ChatGPT to categorize these unique SOC Titles into 10 categories so that it is easier for us to work on.

In [ ]:
unique_soc_titles = data['SOC_TITLE'].unique()
soc_titles_df = pd.DataFrame({'SOC_TITLE': unique_soc_titles})
soc_titles_df.to_csv('unique_soc_titles.csv', index=False)


### Download CSV generated by ChatGPT for categorization

In [ ]:
import os
import requests

def download_csv(url, filename):
    """Download CSV if it doesn't already exist in current directory"""
    if os.path.isfile(filename):
        print(f"File '{filename}' already exists.")
        return False

    response = requests.get(url)
    with open(filename, 'wb') as file:
        file.write(response.content)
    print(f"Downloaded '{filename}'")
    return True

download_csv(
    "https://gist.githubusercontent.com/naman-gupta99/a4fd363fd0c3ef6f0c404ac875d51b54/raw/1b6bfa1f21b55152c6c9411fc642da6622c9b8fd/soc_title_to_categories.csv",
    "soc_title_to_categories.csv")

soc_title_to_categories = pd.read_csv("soc_title_to_categories.csv")
soc_title_to_categories

In [ ]:
data = pd.merge(data, soc_title_to_categories, on='SOC_TITLE', how='left')

data.head()['Category_Name']

## Include SOC_CATEGORIES to DATASET

# Geomapping

### Employer State Frequency

In [ ]:
emp_state_freq = data['EMPLOYER_STATE'].value_counts().reset_index()

fig = px.choropleth(
    emp_state_freq,
    locations='EMPLOYER_STATE',
    locationmode="USA-states",
    color='count',
    scope="usa",
    color_continuous_scale='Viridis'
)

fig.update_layout(title_text='Frequency of Employers by State', geo=dict(lakecolor='rgb(255, 255, 255)'))

fig.show()

### Agent Attorney State Frequency

In [ ]:
attorney_state_freq = data['AGENT_ATTORNEY_STATE'].value_counts().reset_index()

fig = px.choropleth(
    attorney_state_freq,
    locations='AGENT_ATTORNEY_STATE',
    locationmode="USA-states",
    color='count',
    scope="usa",
    color_continuous_scale='Viridis'
)

fig.update_layout(title_text='Frequency of Agent Attorney by State', geo=dict(lakecolor='rgb(255, 255, 255)'))

fig.show()

### Worksite state frequency

In [ ]:
worksite_state_frequency = data['WORKSITE_STATE'].value_counts().reset_index()

fig = px.choropleth(
    worksite_state_frequency,
    locations='WORKSITE_STATE',
    locationmode="USA-states",
    color='count',
    scope="usa",
    color_continuous_scale='Viridis'
)

fig.update_layout(title_text='Frequency of Worksite by State', geo=dict(lakecolor='rgb(255, 255, 255)'))

fig.show()

# Visa Application Frequency over Time

In [ ]:
time_freq = data.groupby([pd.Grouper(key='RECEIVED_DATE', freq='M')]).size().reset_index(name='counts')

fig = px.line(state_time_freq, x='RECEIVED_DATE', y='counts', title='Visa Application Frequency over Time')
fig.show()

## By State

In [ ]:
state_time_freq = data.groupby([pd.Grouper(key='RECEIVED_DATE', freq='M'), 'EMPLOYER_STATE']).size().reset_index(name='counts')

fig = px.line(state_time_freq, x='RECEIVED_DATE', y='counts', color='EMPLOYER_STATE', title='Visa Application Frequency over Time by State')
fig.show()

## By Category

In [ ]:
data['RECEIVED_DATE'] = pd.to_datetime(data['RECEIVED_DATE'])

category_time_freq = data.groupby([pd.Grouper(key='RECEIVED_DATE', freq='M'), 'Category_Name']).size().reset_index(name='counts')

fig = px.line(category_time_freq, x='RECEIVED_DATE', y='counts', color='Category_Name', title='Visa Application Frequency over Time by Category')
fig.show()